In [ ]:
import pandas as pd
import sqlite3
from sqlalchemy import create_engine
from sqlalchemy import text

In [49]:
flights_data = pd.read_csv('../../Flight_Data/flight_data_sample.csv')

In [ ]:
# sampled_df = (
#     flights_data
#     .groupby('month', group_keys=False)
#     .apply(lambda x: x.sample(frac=0.01, random_state=42))
# )

# sampled_df = sampled_df.reset_index(drop=True)
# sampled_df.to_csv('./sample_test.csv', index=False)

In [51]:
print(flights_data.columns)

Index(['year', 'month', 'day_of_month', 'day_of_week', 'fl_date',
       'op_unique_carrier', 'op_carrier_fl_num', 'origin_city_name',
       'origin_state_nm', 'dest_city_name', 'dest_state_nm', 'crs_dep_time',
       'dep_time', 'dep_delay', 'crs_arr_time', 'arr_time', 'arr_delay',
       'cancelled', 'crs_elapsed_time', 'actual_elapsed_time', 'air_time',
       'carrier_delay', 'weather_delay', 'nas_delay', 'security_delay',
       'late_aircraft_delay'],
      dtype='object')


In [52]:
engine = create_engine('sqlite:///flights.db')

In [53]:
flights_data.to_sql('flights', engine, if_exists='replace', index=False)

1393053

In [54]:
head_query = "SELECT * FROM flights LIMIT 10;"
head_flights = pd.read_sql(head_query, engine)
print(head_flights)

   year  month  day_of_month  day_of_week     fl_date op_unique_carrier  \
0  2024      1             4            4  2024-01-04                WN   
1  2024      1            21            7  2024-01-21                OO   
2  2024      1            26            5  2024-01-26                WN   
3  2024      1            22            1  2024-01-22                MQ   
4  2024      1             7            7  2024-01-07                WN   
5  2024      1            15            1  2024-01-15                NK   
6  2024      1             5            5  2024-01-05                YX   
7  2024      1            14            7  2024-01-14                YX   
8  2024      1            24            3  2024-01-24                WN   
9  2024      1            10            3  2024-01-10                WN   

   op_carrier_fl_num       origin_city_name origin_state_nm  \
0              673.0        Kansas City, MO        Missouri   
1             5458.0            Chicago, IL     

### Adding Delay_in_Minutes Column

In [ ]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE flights
        ADD COLUMN delay_in_minutes INTEGER;
    """))
    
    conn.execute(text("""
        UPDATE flights
        SET delay_in_minutes = arr_delay + dep_delay
    """))

In [58]:
head_flights = pd.read_sql(head_query, engine)
print(head_flights)

   year  month  day_of_month  day_of_week     fl_date op_unique_carrier  \
0  2024      1             4            4  2024-01-04                WN   
1  2024      1            21            7  2024-01-21                OO   
2  2024      1            26            5  2024-01-26                WN   
3  2024      1            22            1  2024-01-22                MQ   
4  2024      1             7            7  2024-01-07                WN   
5  2024      1            15            1  2024-01-15                NK   
6  2024      1             5            5  2024-01-05                YX   
7  2024      1            14            7  2024-01-14                YX   
8  2024      1            24            3  2024-01-24                WN   
9  2024      1            10            3  2024-01-10                WN   

   op_carrier_fl_num       origin_city_name origin_state_nm  \
0              673.0        Kansas City, MO        Missouri   
1             5458.0            Chicago, IL     

### Adding Is_Delay Flag Column

In [59]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE flights
        ADD COLUMN IS_Delay BOOL;
    """))
    
    conn.execute(text("""
        UPDATE flights
        SET IS_Delay = CASE 
        WHEN COALESCE(delay_in_minutes,0) >= 10 THEN 1
        ELSE 0
        END;
    """))

In [61]:
with engine.begin() as conn:
    conn.execute(text("""
        UPDATE flights
        SET 
            carrier_delay = CASE WHEN is_delay = 0 THEN 0 ELSE carrier_delay END,
            weather_delay = CASE WHEN is_delay = 0 THEN 0 ELSE weather_delay END,
            nas_delay = CASE WHEN is_delay = 0 THEN 0 ELSE nas_delay END,
            security_delay = CASE WHEN is_delay = 0 THEN 0 ELSE security_delay END,
            late_aircraft_delay = CASE WHEN is_delay = 0 THEN 0 ELSE late_aircraft_delay END;
    """))

    

### Creating Season Columns

In [63]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE flights
        ADD COLUMN Season TEXT;
    """))
    
    conn.execute(text("""
        UPDATE flights
        SET Season = CASE WHEN month IN (12, 1, 2) THEN 'Winter'
        WHEN month IN (3, 4, 5) THEN 'Spring'
        WHEN month IN (6, 7, 8) THEN 'Summer'   
        WHEN month IN (9, 10, 11) THEN 'Fall'
        ELSE 'Unknown'
        END;
    """))

In [64]:
head_flights = pd.read_sql(head_query, engine)
print(head_flights)

   year  month  day_of_month  day_of_week     fl_date op_unique_carrier  \
0  2024      1             4            4  2024-01-04                WN   
1  2024      1            21            7  2024-01-21                OO   
2  2024      1            26            5  2024-01-26                WN   
3  2024      1            22            1  2024-01-22                MQ   
4  2024      1             7            7  2024-01-07                WN   
5  2024      1            15            1  2024-01-15                NK   
6  2024      1             5            5  2024-01-05                YX   
7  2024      1            14            7  2024-01-14                YX   
8  2024      1            24            3  2024-01-24                WN   
9  2024      1            10            3  2024-01-10                WN   

   op_carrier_fl_num       origin_city_name origin_state_nm  \
0              673.0        Kansas City, MO        Missouri   
1             5458.0            Chicago, IL     

In [ ]:
df = pd.read_sql("SELECT * FROM flights;", engine)


In [66]:
df.to_csv('./Flight_data_sample.csv', index=False)